# Stabilität des Modellvergleichs

Wir vergleichen die Merkmalversionen 1 und 2 der logistischen Regression auf fünf nach Personen getrennten Validierungsaufteilungen. Bewertet werden Average Precision und Recall bei 60 % und 70 % vorgeschlagenen Kurseinträgen. Der ursprünglich beiseitegelegte Testsatz ist von dieser Prüfung ausgeschlossen.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)

projektordner = Path(
    r"C:\Users\Tony Willert\OneDrive\Weiterbildung\PortfolioProjekt"
)
ordner = projektordner / "data" / "processed"

v1 = pd.read_csv(ordner / "oulad_tag14_modellbasis.csv")
v2 = pd.read_csv(ordner / "oulad_tag14_modellbasis_v2.csv")

# Nur die Features dürfen sich zwischen V1 und V2 unterscheiden.
identitaet = [
    "id_student", "code_module", "code_presentation",
    "spaeter_abgemeldet"
]
assert v1[identitaet].equals(v2[identitaet])

# Genau dieselbe Aufteilung wie in Notebook 02 wiederherstellen.
aufteilung = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
train_index, _ = next(
    aufteilung.split(v1, groups=v1["id_student"])
)

train_v1 = v1.iloc[train_index].copy()
train_v2 = v2.iloc[train_index].copy()

print("Training V1:", train_v1.shape)
print("Training V2:", train_v2.shape)
print("Gleiche Personen und Ergebnisse: ja")

Training V1: (22371, 8)
Training V2: (22371, 9)
Gleiche Personen und Ergebnisse: ja


In [5]:
def baue_logistische_regression(aktive_spalten):
    klickspalten = [
        "klicks_tag_0_bis_6",
        "klicks_tag_7_bis_14",
    ]
    zahlenspalten = ["anmeldetag"] + aktive_spalten
    kategoriespalten = ["code_module", "code_presentation"]

    klickverarbeitung = Pipeline([
        ("log", FunctionTransformer(np.log1p)),
        ("skalierung", StandardScaler()),
    ])

    vorverarbeitung = ColumnTransformer([
        ("klicks", klickverarbeitung, klickspalten),
        ("zahlen", StandardScaler(), zahlenspalten),
        ("kategorien", OneHotEncoder(handle_unknown="ignore"),
         kategoriespalten),
    ])

    modell = Pipeline([
        ("vorverarbeitung", vorverarbeitung),
        ("klassifikation", LogisticRegression(max_iter=1000)),
    ])

    merkmale = klickspalten + zahlenspalten + kategoriespalten
    return modell, merkmale

In [6]:
falten = list(
    GroupKFold(n_splits=5).split(
        train_v1,
        train_v1["spaeter_abgemeldet"],
        groups=train_v1["id_student"],
    )
)

varianten = [
    ("V1", train_v1, ["aktive_tage_bis_14"]),
    ("V2", train_v2, [
        "aktive_tage_0_bis_6",
        "aktive_tage_7_bis_14",
    ]),
]

ergebnisse = []

for name, daten_version, aktive_spalten in varianten:
    for fold_nr, (fit_index, valid_index) in enumerate(falten, start=1):
        fit_daten = daten_version.iloc[fit_index]
        valid_daten = daten_version.iloc[valid_index]

        # Für jeden Fold ein neues Modell nur an fit_daten anpassen.
        kandidat, merkmale = baue_logistische_regression(
            aktive_spalten
        )
        kandidat.fit(
            fit_daten[merkmale],
            fit_daten["spaeter_abgemeldet"],
        )

        y = valid_daten["spaeter_abgemeldet"].to_numpy()
        werte = kandidat.predict_proba(valid_daten[merkmale])[:, 1]
        reihenfolge = np.argsort(-werte, kind="stable")

        zeile = {
            "Version": name,
            "Fold": fold_nr,
            "AP": average_precision_score(y, werte),
            "Abbruchanteil": y.mean(),
        }

        for anteil in [0.60, 0.70]:
            anzahl = round(len(y) * anteil)
            treffer = y[reihenfolge[:anzahl]].sum()
            zeile[f"Recall_{int(anteil * 100)}"] = (
                treffer / y.sum()
            )

        ergebnisse.append(zeile)

cv_ergebnisse = pd.DataFrame(ergebnisse)

print("Ergebnisse der einzelnen Folds:")
display(cv_ergebnisse.round(3))

print("Mittelwert und Streuung:")
display(
    cv_ergebnisse.groupby("Version")[
        ["AP", "Recall_60", "Recall_70"]
    ].agg(["mean", "std"]).round(3)
)

Ergebnisse der einzelnen Folds:


,Version,Fold,AP,Abbruchanteil,Recall_60,Recall_70
0,V1,1,0.324,0.196,0.760,0.825
1,V1,2,0.324,0.199,0.767,0.843
2,V1,3,0.340,0.213,0.779,0.847
3,V1,4,0.308,0.192,0.758,0.849
4,V1,5,0.333,0.206,0.772,0.846
5,V2,1,0.325,0.196,0.759,0.824
6,V2,2,0.324,0.199,0.768,0.843
7,V2,3,0.340,0.213,0.776,0.849
8,V2,4,0.308,0.192,0.761,0.848
9,V2,5,0.334,0.206,0.773,0.845


Mittelwert und Streuung:


AP        Recall_60        Recall_70      
          mean    std      mean    std      mean   std
Version                                               
V1       0.326  0.012     0.767  0.008     0.842  0.01
V2       0.326  0.012     0.767  0.008     0.842  0.01

## Ergebnis des Merkmalvergleichs

Die Aufteilung aktiver Tage in zwei Zeitfenster verbessert die logistische Regression über fünf nach Personen getrennte Folds nicht erkennbar. Für die folgenden Modellvergleiche verwenden wir daher Version 1 mit der Gesamtzahl aktiver Tage. Der beiseitegelegte Testsatz wurde nicht verwendet.

### Zwei weitere Modelle auf denselben fünf Folds

Als Referenz verwenden wir eine logistische Regression, die nur Modul und
Kursdurchlauf kennt. Daneben testen wir Histogram Gradient Boosting mit den
V1-Merkmalen. Wir vergleichen Average Precision sowie den Recall, wenn 60 %
oder 70 % der Kurseinträge einen Hinweis erhalten.

In [7]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder

kurs_merkmale = ["code_module", "code_presentation"]
klick_merkmale = ["klicks_tag_0_bis_6", "klicks_tag_7_bis_14"]
weitere_merkmale = ["anmeldetag", "aktive_tage_bis_14"]
v1_merkmale = kurs_merkmale + klick_merkmale + weitere_merkmale

# Referenz: Das Modell sieht ausschließlich Modul und Kursdurchlauf.
kursmodell = Pipeline([
    ("vorverarbeitung", OneHotEncoder(handle_unknown="ignore")),
    ("klassifikation", LogisticRegression(max_iter=1000)),
])

# Beim Boosting stehen die beiden Kurskategorien an Position 0 und 1.
# Die Klickzahlen werden logarithmiert; die übrigen Zahlen bleiben unverändert.
boost_vorverarbeitung = ColumnTransformer([
    (
        "kategorien",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=np.nan,
        ),
        kurs_merkmale,
    ),
    ("klicks", FunctionTransformer(np.log1p), klick_merkmale),
    ("zahlen", "passthrough", weitere_merkmale),
])

boostmodell = Pipeline([
    ("vorverarbeitung", boost_vorverarbeitung),
    ("klassifikation", HistGradientBoostingClassifier(
        categorical_features=[0, 1],
        max_iter=100,
        max_leaf_nodes=15,
        learning_rate=0.05,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )),
])

kandidaten = [
    ("Nur Kurskontext", kursmodell, kurs_merkmale),
    ("Histogram Gradient Boosting V1", boostmodell, v1_merkmale),
]

In [8]:
from sklearn.model_selection import GroupKFold

# Teilt die Trainingsdaten in fünf Folds auf.
# Alle Kurseinträge derselben Person bleiben jeweils zusammen.
falten = list(
    GroupKFold(n_splits=5).split(
        train_v1,
        train_v1["spaeter_abgemeldet"],
        groups=train_v1["id_student"],
    )
)

print("Anzahl Folds:", len(falten))

assert len(falten) == 5

ergebnisse_zusatz = []

for verfahren, modell, merkmale in kandidaten:
    for fold_nr, (train_idx, valid_idx) in enumerate(falten, start=1):
        train_daten = train_v1.iloc[train_idx]
        valid_daten = train_v1.iloc[valid_idx]

        # Sicherheitscheck: Keine Person darf auf beiden Seiten des Folds stehen.
        assert set(train_daten["id_student"]).isdisjoint(
            set(valid_daten["id_student"])
        )

        # Jeder Fold bekommt ein frisch trainiertes Modell.
        kandidat = clone(modell)
        kandidat.fit(
            train_daten[merkmale],
            train_daten["spaeter_abgemeldet"],
        )

        y_valid = valid_daten["spaeter_abgemeldet"].to_numpy(dtype=int)
        werte = kandidat.predict_proba(valid_daten[merkmale])[:, 1]
        rangfolge = np.argsort(-werte, kind="stable")

        ergebnis = {
            "Verfahren": verfahren,
            "Fold": fold_nr,
            "AP": average_precision_score(y_valid, werte),
            "Abbruchanteil": y_valid.mean(),
        }

        # Dieselbe Rundung und Rangfolge wie im bisherigen V1/V2-Vergleich.
        for anteil in (60, 70):
            anzahl_hinweise = round(len(y_valid) * anteil / 100)
            erkannte_abmeldungen = y_valid[
                rangfolge[:anzahl_hinweise]
            ].sum()

            ergebnis[f"Recall_{anteil}"] = (
                erkannte_abmeldungen / y_valid.sum()
            )

        ergebnisse_zusatz.append(ergebnis)

cv_zusatz = pd.DataFrame(ergebnisse_zusatz)

# Bisherige V1/V2-Ergebnisse und neue Kandidaten gemeinsam anzeigen.
cv_alt = cv_ergebnisse.copy()
cv_alt["Verfahren"] = cv_alt["Version"].map({
    "V1": "Logistische Regression V1",
    "V2": "Logistische Regression V2",
})

spalten = [
    "Verfahren", "Fold", "AP", "Abbruchanteil",
    "Recall_60", "Recall_70",
]
vergleich_cv = pd.concat(
    [cv_alt[spalten], cv_zusatz[spalten]],
    ignore_index=True,
)

display(
    vergleich_cv.groupby("Verfahren", sort=False)[
        ["AP", "Recall_60", "Recall_70"]
    ].agg(["mean", "std"]).round(3)
)

display(
    vergleich_cv.pivot(
        index="Fold",
        columns="Verfahren",
        values="AP",
    ).round(3)
)

Anzahl Folds: 5


AP        Recall_60        Recall_70       
                                 mean    std      mean    std      mean    std
Verfahren                                                                     
Logistische Regression V1       0.326  0.012     0.767  0.008     0.842  0.010
Logistische Regression V2       0.326  0.012     0.767  0.008     0.842  0.010
Nur Kurskontext                 0.280  0.013     0.740  0.012     0.814  0.011
Histogram Gradient Boosting V1  0.333  0.012     0.776  0.012     0.840  0.007

Verfahren,Histogram Gradient Boosting V1,Logistische Regression V1,Logistische Regression V2,Nur Kurskontext
Fold,,,,
1,0.325,0.324,0.325,0.280
2,0.342,0.324,0.324,0.287
3,0.346,0.340,0.340,0.295
4,0.318,0.308,0.308,0.261
5,0.336,0.333,0.334,0.278


### Zwischenfazit zum Modellvergleich

Die individuellen Aktivitätsmerkmale verbessern die Vorhersage gegenüber
dem reinen Kurskontext (AP 0,326 statt 0,280 für die logistische Regression V1).
Histogram Gradient Boosting erreicht in allen fünf Folds eine etwas höhere AP
(0,333), verbessert den Recall bei 70 % Hinweisen jedoch nicht (0,840
gegenüber 0,842).

Da das Projekt einen hohen Recall priorisiert, bleibt die logistische
Regression V1 vorerst die einfache Referenz. Als nächsten begrenzten Versuch
prüfen wir die Zeit seit der letzten VLE-Aktivität an Tag 14. Danach
entscheiden wir über das finale Modell; der Testsatz bleibt bis dahin ungenutzt.

In [9]:
from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

schluessel = ["code_module", "code_presentation", "id_student"]

# Nur das neue Merkmal an die bestehende Trainingsgruppe hängen.
# So bleiben Personen und Reihenfolge der fünf Folds exakt dieselben.
v3 = pd.read_csv(ordner / "oulad_tag14_modellbasis_v3.csv")
train_v3 = train_v1.merge(
    v3[schluessel + ["tage_seit_letzter_aktivitaet"]],
    on=schluessel,
    how="left",
    validate="one_to_one",
)

assert len(train_v3) == len(train_v1)
assert train_v3[schluessel].equals(
    train_v1[schluessel].reset_index(drop=True)
)
assert train_v3["tage_seit_letzter_aktivitaet"].notna().all()

ergebnisse_v3 = []

for fold_nr, (train_idx, valid_idx) in enumerate(falten, start=1):
    train_daten = train_v3.iloc[train_idx]
    valid_daten = train_v3.iloc[valid_idx]

    # V1-Modell um das neue Zahlenmerkmal erweitern und frisch trainieren.
    modell, merkmale = baue_logistische_regression([
        "aktive_tage_bis_14",
        "tage_seit_letzter_aktivitaet",
    ])
    modell.fit(
        train_daten[merkmale],
        train_daten["spaeter_abgemeldet"],
    )

    y_valid = valid_daten["spaeter_abgemeldet"].to_numpy(dtype=int)
    werte = modell.predict_proba(valid_daten[merkmale])[:, 1]
    rangfolge = np.argsort(-werte, kind="stable")

    ergebnis = {
        "Version": "V3",
        "Fold": fold_nr,
        "AP": average_precision_score(y_valid, werte),
    }

    for anteil in (60, 70):
        anzahl_hinweise = round(len(y_valid) * anteil / 100)
        ergebnis[f"Recall_{anteil}"] = (
            y_valid[rangfolge[:anzahl_hinweise]].sum()
            / y_valid.sum()
        )

    ergebnisse_v3.append(ergebnis)

# Nur V1 und V3 gegenüberstellen; die früheren Ergebnisse bleiben erhalten.
vergleich_v3 = pd.concat([
    cv_ergebnisse.loc[
        cv_ergebnisse["Version"].eq("V1"),
        ["Version", "Fold", "AP", "Recall_60", "Recall_70"],
    ],
    pd.DataFrame(ergebnisse_v3),
], ignore_index=True)

display(
    vergleich_v3.groupby("Version")[
        ["AP", "Recall_60", "Recall_70"]
    ].agg(["mean", "std"]).round(3)
)

display(
    vergleich_v3.pivot(
        index="Fold",
        columns="Version",
        values=["AP", "Recall_60", "Recall_70"],
    ).round(3)
)

AP        Recall_60        Recall_70      
          mean    std      mean    std      mean   std
Version                                               
V1       0.326  0.012     0.767  0.008     0.842  0.01
V3       0.326  0.012     0.768  0.008     0.842  0.01

AP        Recall_60        Recall_70       
Version     V1     V3        V1     V3        V1     V3
Fold                                                   
1        0.324  0.324     0.760  0.761     0.825  0.825
2        0.324  0.324     0.767  0.765     0.843  0.845
3        0.340  0.340     0.779  0.779     0.847  0.846
4        0.308  0.307     0.758  0.760     0.849  0.849
5        0.333  0.332     0.772  0.774     0.846  0.846

### V3: Zeit seit der letzten VLE-Aktivität

V3 ergänzt V1 um die Anzahl der Tage seit dem letzten VLE-Klick bis
einschließlich Tag 14. Der Wert 15 kennzeichnet Kurseinträge ohne
beobachteten Klick in diesem Zeitraum.

Bei der logistischen Regression verändert V3 die Ergebnisse kaum
(AP 0,326 für beide Versionen; Recall bei 70 % Hinweisen 0,842).
Abschließend prüfen wir, ob Gradient Boosting das neue Merkmal besser nutzt.

In [10]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.metrics import average_precision_score

import numpy as np
import pandas as pd

kurs_merkmale = ["code_module", "code_presentation"]
klick_merkmale = ["klicks_tag_0_bis_6", "klicks_tag_7_bis_14"]
zahlen_v3 = [
    "anmeldetag",
    "aktive_tage_bis_14",
    "tage_seit_letzter_aktivitaet",
]
merkmale_v3 = kurs_merkmale + klick_merkmale + zahlen_v3

# Dieselbe Aufbereitung wie beim Boosting V1, ergänzt um das neue Merkmal.
vorverarbeitung_v3 = ColumnTransformer([
    (
        "kategorien",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=np.nan,
        ),
        kurs_merkmale,
    ),
    ("klicks", FunctionTransformer(np.log1p), klick_merkmale),
    ("zahlen", "passthrough", zahlen_v3),
])

boostmodell_v3 = Pipeline([
    ("vorverarbeitung", vorverarbeitung_v3),
    # Übernimmt exakt die bisherigen Boosting-Einstellungen.
    ("klassifikation", clone(boostmodell.named_steps["klassifikation"])),
])

ergebnisse_boost_v3 = []

for fold_nr, (train_idx, valid_idx) in enumerate(falten, start=1):
    train_daten = train_v3.iloc[train_idx]
    valid_daten = train_v3.iloc[valid_idx]

    # Pro Fold ein neues Modell fitten; keine Daten des Folds vorab verwenden.
    kandidat = clone(boostmodell_v3)
    kandidat.fit(
        train_daten[merkmale_v3],
        train_daten["spaeter_abgemeldet"],
    )

    y_valid = valid_daten["spaeter_abgemeldet"].to_numpy(dtype=int)
    werte = kandidat.predict_proba(valid_daten[merkmale_v3])[:, 1]
    rangfolge = np.argsort(-werte, kind="stable")

    ergebnis = {
        "Verfahren": "Histogram Gradient Boosting V3",
        "Fold": fold_nr,
        "AP": average_precision_score(y_valid, werte),
    }

    for anteil in (60, 70):
        anzahl_hinweise = round(len(y_valid) * anteil / 100)
        ergebnis[f"Recall_{anteil}"] = (
            y_valid[rangfolge[:anzahl_hinweise]].sum()
            / y_valid.sum()
        )

    ergebnisse_boost_v3.append(ergebnis)

vergleich_boost = pd.concat([
    cv_zusatz.loc[
        cv_zusatz["Verfahren"].eq("Histogram Gradient Boosting V1"),
        ["Verfahren", "Fold", "AP", "Recall_60", "Recall_70"],
    ],
    pd.DataFrame(ergebnisse_boost_v3),
], ignore_index=True)

display(
    vergleich_boost.groupby("Verfahren")[
        ["AP", "Recall_60", "Recall_70"]
    ].agg(["mean", "std"]).round(3)
)

display(
    vergleich_boost.pivot(
        index="Fold",
        columns="Verfahren",
        values=["AP", "Recall_60", "Recall_70"],
    ).round(3)
)


AP        Recall_60        Recall_70       
                                 mean    std      mean    std      mean    std
Verfahren                                                                     
Histogram Gradient Boosting V1  0.333  0.012     0.776  0.012     0.840  0.007
Histogram Gradient Boosting V3  0.332  0.010     0.771  0.011     0.841  0.010

AP                                 \
Verfahren Histogram Gradient Boosting V1 Histogram Gradient Boosting V3   
Fold                                                                      
1                                  0.325                          0.324   
2                                  0.342                          0.336   
3                                  0.346                          0.342   
4                                  0.318                          0.319   
5                                  0.336                          0.339   

                               Recall_60                                 \
Verfahren Histogram Gradient Boosting V1 Histogram Gradient Boosting V3   
Fold                                                                      
1                                  0.757                          0.753   
2                                  0.787                          0.781   
3                                  0.778                          0.772   
4                                  0.775                          0.771   
5                                  0.782                          0.777   

                               Recall_70                                 
Verfahren Histogram Gradient Boosting V1 Histogram Gradient Boosting V3  
Fold                                                                     
1                                  0.829                          0.827  
2                                  0.843                          0.835  
3                                  0.847                          0.842  
4                                  0.843                          0.851  
5                                  0.839                          0.848